# Data generation script

In [1]:
import matplotlib.pylab as plt
import numpy as np
import scipy
import soundfile as sf
 import os
import tqdm
from os.path import join as pjoin
import scipy.signal as sig

In [2]:
np.random.seed(0)

In [3]:
FS = 48000

# We generate:
* BL (Baseline, General Purpose RIRs)
* D1 (BL + D1, PA Rirs with basic polarity)
* test set of D1
* D2 (BL+D1+D2, Pusing each capsule of the Zylia)
* D3 (BL+D1+D2+D3, adding beamforming)
* D4 (BL+D1+D2+D3 permuting PAs)

# BL Baseline, General Purpose RIRs

We use RIRS from ACE Corpus + MIT IR Survey + SLR28 (RWCP and AIR)

In [4]:
path = '/media/diskA/enric/ACE_Corpus_RIRN_Single'

In [5]:
outpath = '/media/diskA/enric/parirset/b1_gp'
if not os.path.exists(outpath):
    os.makedirs(outpath)

In [6]:
rooms = os.listdir(path)

real_idx = 0
rirpaths = []
for room in rooms:
    positions = os.listdir(pjoin(path, room))
    for position in positions:
        wavs = os.listdir(pjoin(pjoin(path, room), position))
        rirs = [k for k in wavs if 'RIR' in k]
        for rir in rirs:
            data, fs = sf.read(pjoin(pjoin(pjoin(path, room), position), rir))
            if fs != FS:
                gcd = np.gcd(fs, FS)
                up = FS // gcd
                down = fs // gcd
                # Resample
                data_resampled = sig.resample_poly(data, up, down)
            else:
                data_resampled = data
            data_resampled = np.tile(data_resampled, (2,1))
            #if data_resampled.shape[0] != 2:
            #    print(rir)
            #    break
            #rirpaths.append(pjoin(pjoin(pjoin(path, room), position), rir))
            sf.write(pjoin(outpath, f"{real_idx:04d}"+'_GP_ACE.wav'), data_resampled.T, FS)
            real_idx +=1

In [7]:
'''
air_path = '/media/diskA/enric/AIR_1_4'
air_rirs = os.listdir(air_path)

for air_rir in air_rirs:
    try:
        data = scipy.io.loadmat(pjoin(air_path, air_rir))['h_air']
        if data.shape[0] == 1:
            data = np.tile(data, (2,1))
        sf.write(pjoin(outpath, f"{real_idx:04d}"+'_GP_AIR.wav'), data.T, FS)
        real_idx +=1
    except:
        print('corrupt file')
''';

In [8]:
mit_path = '/media/diskA/enric/MIT_IR_Survey'


In [9]:
mit_rirs = os.listdir(mit_path)

In [10]:
for mit_rir in mit_rirs:
    data, fs = sf.read(pjoin(mit_path, mit_rir))
    if fs != FS:
        gcd = np.gcd(fs, FS)
        up = FS // gcd
        down = fs // gcd
        # Resample
        data_resampled = sig.resample_poly(data, up, down)
    else:
        data_resampled = data
    data_resampled = np.tile(data_resampled, (2,1))
    sf.write(pjoin(outpath, f"{real_idx:04d}"+'_GP_MIT.wav'), data_resampled.T, FS)
    real_idx +=1

In [11]:
dns_path = '/media/diskA/enric/SLR28/RIRS_NOISES/real_rirs_isotropic_noises'
dns_rirs = os.listdir(dns_path)

In [12]:
for dns_rir in dns_rirs:
    data, fs = sf.read(pjoin(dns_path, dns_rir))
    if fs != FS:
        gcd = np.gcd(fs, FS)
        up = FS // gcd
        down = fs // gcd
        # Resample
        data_resampled = sig.resample_poly(data, up, down)
    else:
        data_resampled = data
    if len(data_resampled.shape) < 2:
        data_resampled = np.tile(data_resampled, (2,1)).T

    else:

        data_resampled = data_resampled[:,:2]
    sf.write(pjoin(outpath, f"{real_idx:04d}"+'_GP_SLR28.wav'), data_resampled, FS)
    real_idx +=1

## D1, PA RIRs with basic data augmentation

We first split 80% and 20% for testing, and augment 80% with
* whole RIR polarity inversion
* DRR polarity inversion
* DRR random scaling

In [13]:
def stereo_conv(audio, rir):
    #shape is CHANNELS, SAMPLE
    l = scipy.signal.fftconvolve(audio[0,:], rir[0,:], 'valid')
    r = scipy.signal.fftconvolve(audio[1,:], rir[1,:], 'valid')        
    return np.array([l, r])

def power(signal):
    return np.mean(signal**2)

def plot_stereo(signal):
    plt.plot(signal[0,:])
    plt.plot(signal[1,:])



def synch_sigs(sig1,sig2):
    sig1_out=np.zeros(sig1.shape)
    sig2_out=np.zeros(sig2.shape)
    corr = scipy.signal.correlate(sig1, sig2, 'full')
    lag = scipy.signal.correlation_lags(len(sig1), len(sig2), mode='full')[np.argmax(corr)]
    if lag > 0:
        sig2=sig2[0:-lag]
        sig1=sig1[lag:]
    elif lag < 0:
        sig2=sig2[-lag:]
        sig1=sig1[0:lag]

    sig1_out[:sig1.shape[0]]=sig1
    sig2_out[:sig2.shape[0]]=sig2
    return sig1_out,sig2_out, lag




def design_high_shelf(fs, f0, gain_db, Q=0.707):
    """
    fs      : Sampling frequency [Hz]
    f0      : Shelf corner frequency [Hz]
    gain_db : Gain at high frequencies [dB] (negative = cut)
    Q       : Quality factor (controls transition slope)
    """
    A = 10**(gain_db/40)  # convert dB gain to linear
    w0 = 2*np.pi*f0/fs
    alpha = np.sin(w0)/(2*Q)
    cosw0 = np.cos(w0)

    # RBJ cookbook high-shelf coefficients
    b0 =    A*((A+1) + (A-1)*cosw0 + 2*np.sqrt(A)*alpha)
    b1 = -2*A*((A-1) + (A+1)*cosw0)
    b2 =    A*((A+1) + (A-1)*cosw0 - 2*np.sqrt(A)*alpha)
    a0 =       (A+1) - (A-1)*cosw0 + 2*np.sqrt(A)*alpha
    a1 =  2*((A-1) - (A+1)*cosw0)
    a2 =       (A+1) - (A-1)*cosw0 - 2*np.sqrt(A)*alpha

    # Normalize coefficients
    b = np.array([b0, b1, b2]) / a0
    a = np.array([1, a1/a0, a2/a0])

    # Convert to SOS for stability
    sos = sig.tf2sos(b, a)
    return sos

In [14]:
MPATH = '/media/diskA/enric/parirset'    
osweep, fs = sf.read('sweep.wav')
T = 6          # time duration of the sweep
AXIS = np.linspace(0,T,fs*T)  # time axis
L = T/np.log(20000/20)                   

test_track, fs3 = sf.read('test_track.wav')
TEST_TRACK = test_track.T

SWEEP = osweep[:T*fs, 0]
DATA_PATH = pjoin(MPATH, 'original_recordings')
BF_PATH = pjoin(MPATH, 'beamformed_recordings')

RIRS_PATH = pjoin(MPATH, 'rirs')
SAMPLES_PATH = pjoin(MPATH, 'samples')

FS = 48000

SOS = design_high_shelf(fs, 3500, -4, Q=0.5)

In [15]:
def stereo_conv(audio, rir):
    #shape is CHANNELS, SAMPLE
    l = scipy.signal.fftconvolve(audio[0,:], rir[0,:], 'valid')
    r = scipy.signal.fftconvolve(audio[1,:], rir[1,:], 'valid')        
    return np.array([l, r])

def power(signal):
    return np.mean(signal**2)

def plot_stereo(signal):
    plt.plot(signal[0,:])
    plt.plot(signal[1,:])



def synch_sigs(sig1,sig2):
    sig1_out=np.zeros(sig1.shape)
    sig2_out=np.zeros(sig2.shape)
    corr = scipy.signal.correlate(sig1, sig2, 'full')
    lag = scipy.signal.correlation_lags(len(sig1), len(sig2), mode='full')[np.argmax(corr)]
    if lag > 0:
        sig2=sig2[0:-lag]
        sig1=sig1[lag:]
    elif lag < 0:
        sig2=sig2[-lag:]
        sig1=sig1[0:lag]

    sig1_out[:sig1.shape[0]]=sig1
    sig2_out[:sig2.shape[0]]=sig2
    return sig1_out,sig2_out, lag

def get_wet_sweep_mono(room):
    # Take Beyer MM1 recording if available, otherwhise take the first channel from the Zylia mic
    if os.path.exists(os.path.join(os.path.join(DATA_PATH, room), 'beyer.wav')):
        path = os.path.join(os.path.join(DATA_PATH, room), 'beyer.wav')
        iszylia = False
    else:
        path = os.path.join(os.path.join(DATA_PATH, room), 'zylia.wav')
        iszylia = True
    wet_sweep, fs2 = sf.read(path)
    if len(wet_sweep.shape) > 1:
        wet_sweep = wet_sweep[:, 0]
    return wet_sweep, iszylia, fs2
    
def get_wet_sweep_capsules(room, capsule):
    # Take other capsules of the zylia, for data augmentation
    path = os.path.join(os.path.join(DATA_PATH, room), 'zylia.wav')
    wet_sweep, fs2 = sf.read(path)
    wet_sweep = wet_sweep[:, capsule]
    return wet_sweep, True, fs2

def get_wet_sweep_beams(room, capsule):
    # Take Beamformed pre-processed signals, for data augmentation
    try:
        path = os.path.join(os.path.join(BF_PATH, room), 'zyliaBF-'+f"{capsule + 1 :03d}"+'.wav')
        wet_sweep, fs2 = sf.read(path)
    except:
        path = os.path.join(os.path.join(BF_PATH, room), 'zyliaBF-001-'+f"{capsule + 1 :03d}"+'.wav')
        wet_sweep, fs2 = sf.read(path)
    return wet_sweep, True, fs2
    
def gen_out(room, mode, capsule):
    if mode == 'normal':
        wet_sweep, iszylia, fs2 = get_wet_sweep_mono(room)
    if mode == 'capsules':
        wet_sweep, iszylia, fs2 = get_wet_sweep_capsules(room, capsule)
    if mode == 'beams':
        wet_sweep, iszylia, fs2 = get_wet_sweep_beams(room, capsule)

    # resample if needed
    if fs2 != fs:
        print('Resampling '+room)
        gcd = np.gcd(fs2, fs)
        wet_sweep = scipy.signal.resample_poly(wet_sweep, fs // gcd, fs2 // gcd)

    # crop Left and Right PA RIRs
    wet_sweep_L = wet_sweep[:T*fs]
    wet_sweep_R = wet_sweep[2*T*fs:3*T*fs]

    # compute the inverse filter and obtain the rir
    s_inv = np.flip(SWEEP, 0) / np.exp(AXIS/L)
    h_l = scipy.signal.fftconvolve(wet_sweep_L, s_inv)
    h_r = scipy.signal.fftconvolve(wet_sweep_R, s_inv)

    #sync left and right rirs (virtually center the mic in the room)
    nh_l, nh_r, lag = synch_sigs(h_l, h_r)

    
    #use a smoothed envelope for cropping before and after the RIR falls into the noise floor
    rir = np.vstack((nh_l, nh_r))    
    envelope = np.abs(sig.hilbert(rir[1]))
    smoothing = 100
    envelope = sig.fftconvolve(envelope, 1/smoothing * np.ones(smoothing), 'same')
    env_max = np.argmax(envelope)
    noise_floor = envelope[env_max - int(2*fs*0.1) : env_max- int(fs*0.1)]
    noise_floor_avg = np.mean(noise_floor)
    
    # we crop from 100ms before direct sound
    start_idx = env_max - int(fs*0.1)
    
    # apply a fade in for the first 50ms
    rir = rir[:, start_idx:]
    fadein = np.hstack((np.linspace(0, 1, int(env_max - start_idx) // 2), np.ones(rir.shape[1] - int(env_max - start_idx) // 2)))
    rir[0] *= fadein
    rir[1] *= fadein

    # crop after if falls into the noise floor
    envelope = envelope[start_idx:]
    end_idx = np.where(envelope[int(fs*0.1):] < noise_floor_avg)[0][0]
    rir = rir[:, :end_idx]
    thlds = np.where(envelope < noise_floor_avg)[0]
    end_idx = thlds[thlds>int(fs*0.1)][0]
    envelope = envelope[:end_idx]
    rir = rir[:, :end_idx]

    # apply a fade out to avoid clicks
    fadeout = np.hstack((np.ones(rir.shape[1] - int(env_max - start_idx) // 2), np.linspace(1, 0, int(env_max - start_idx) // 2)))
    rir[0] *= fadeout
    rir[1] *= fadeout

    if iszylia:
        #print('EQing RIR. '+room)
        rir = sig.sosfilt(SOS, rir)
    
    #normalize
    global_max = np.max((np.abs(rir[0]), np.abs(rir[1])))
    rir[0] /= np.max(np.abs(rir[0])) / global_max
    rir[1] /= np.max(np.abs(rir[1])) / global_max
    rir /= np.max(np.abs(rir))

    # apply to the test track for generating a sample
    #out = stereo_conv(TEST_TRACK, rir)
    
    #out *= np.sqrt(power(test_track) / power(out)) 
    
    split_idx = np.argmax(rir[0])+int(0.002 * fs)
    
    #return out, rir
    return rir, split_idx

def design_high_shelf(fs, f0, gain_db, Q=0.707):
    """
    Designs a high-shelf filter using RBJ audio EQ cookbook formulas.
    Returns a second-order section (sos) array.
    
    fs      : Sampling frequency [Hz]
    f0      : Shelf corner frequency [Hz]
    gain_db : Gain at high frequencies [dB] (negative = cut)
    Q       : Quality factor (controls transition slope)
    """
    A = 10**(gain_db/40)  # convert dB gain to linear
    w0 = 2*np.pi*f0/fs
    alpha = np.sin(w0)/(2*Q)
    cosw0 = np.cos(w0)

    # RBJ cookbook high-shelf coefficients
    b0 =    A*((A+1) + (A-1)*cosw0 + 2*np.sqrt(A)*alpha)
    b1 = -2*A*((A-1) + (A+1)*cosw0)
    b2 =    A*((A+1) + (A-1)*cosw0 - 2*np.sqrt(A)*alpha)
    a0 =       (A+1) - (A-1)*cosw0 + 2*np.sqrt(A)*alpha
    a1 =  2*((A-1) - (A+1)*cosw0)
    a2 =       (A+1) - (A-1)*cosw0 - 2*np.sqrt(A)*alpha

    # Normalize coefficients
    b = np.array([b0, b1, b2]) / a0
    a = np.array([1, a1/a0, a2/a0])

    # Convert to SOS for stability
    sos = sig.tf2sos(b, a)
    return sos

In [16]:
outpath = '/media/diskA/enric/parirset/d1_original'
if not os.path.exists(outpath):
    os.makedirs(outpath)
    
outpath_test = '/media/diskA/enric/parirset/test'
if not os.path.exists(outpath_test):
    os.makedirs(outpath_test)

In [17]:
rooms = os.listdir(pjoin(MPATH, 'original_recordings'))
rooms.sort()
rooms = rooms[1:-5]

In [18]:
np.random.seed(1)
trainrooms = list(np.random.choice(rooms, int(len(rooms)*0.8), replace=False))

In [19]:
trainrooms.sort()

In [20]:
testrooms = []
for room in rooms:
    if room not in trainrooms:
        testrooms.append(room)

In [21]:
idx = 0
gains = np.hstack((1, np.linspace(0.5, 1.5, 4)))
for room in testrooms:
    try:
        rir, split_id = gen_out(room, 'normal', 0)

        for gain in gains:
            original = np.hstack((rir[:, :split_id], gain * rir[:, split_id:]))
            aug1 = -original 
            aug2 = np.hstack((-1 * rir[:, :split_id], gain * rir[:, split_id:]))
            aug3 = np.hstack((rir[:, :split_id], -gain * rir[:, split_id:]))
        
            sf.write(pjoin(outpath_test, f"{idx:04d}"+'_'+room+'_test.wav'), original.T, samplerate=FS)
            sf.write(pjoin(outpath_test, f"{idx+1:04d}"+'_'+room+'_test.wav'), aug1.T, samplerate=FS)
            sf.write(pjoin(outpath_test, f"{idx+2:04d}"+'_'+room+'_test.wav'), aug2.T, samplerate=FS)
            sf.write(pjoin(outpath_test, f"{idx+3:04d}"+'_'+room+'_test.wav'), aug3.T, samplerate=FS)
            idx += 4
    except:
            print('Error in '+room)

In [22]:
idx = 0
deltas = []
tails = []
for room in trainrooms:
    try:
        rir, split_id = gen_out(room, 'normal', 0)
        deltas.append(rir[:, :split_id])
        tails.append(rir[:, split_id:])
        for gain in gains:
            original = np.hstack((rir[:, :split_id], gain * rir[:, split_id:]))
            aug1 = -original 
            aug2 = np.hstack((-1 * rir[:, :split_id], gain * rir[:, split_id:]))
            aug3 = np.hstack((rir[:, :split_id], -gain * rir[:, split_id:]))
            sf.write(pjoin(outpath, f"{idx:04d}"+'_'+room+'_d1_orig.wav'), original.T, samplerate=FS)
            sf.write(pjoin(outpath, f"{idx+1:04d}"+'_'+room+'_d1_au1.wav'), aug1.T, samplerate=FS)
            sf.write(pjoin(outpath, f"{idx+2:04d}"+'_'+room+'_d1_au2.wav'), aug2.T, samplerate=FS)
            sf.write(pjoin(outpath, f"{idx+3:04d}"+'_'+room+'_d1_au3.wav'), aug3.T, samplerate=FS)
            idx += 4
    except:
            print('Error in '+room)

In [23]:
# finally the K2 RIR:
rir_L, fs2 = sf.read(pjoin(pjoin(MPATH, 'original_recordings'), '40_K2/P1_EQ.wav'))
rir_R, fs2 = sf.read(pjoin(pjoin(MPATH, 'original_recordings'), '40_K2/P2_EQ.wav'))

rir = np.vstack((rir_L, rir_R))

rir = rir[:, np.min((np.argmax(rir[0]), np.argmax(rir[1]))) - int(0.1*fs):]

global_max = np.max((np.abs(rir[0]), np.abs(rir[1])))

rir[0] /= np.max(np.abs(rir[0])) / global_max
rir[1] /= np.max(np.abs(rir[1])) / global_max
rir /= np.max(np.abs(rir))

sf.write(pjoin(outpath, f"{idx+3:04d}"+'_'+'40_K2p1_d1_orig.wav'), rir.T, samplerate=FS)

In [24]:
# finally the K2 RIR:
rir_L, fs2 = sf.read(pjoin(pjoin(MPATH, 'original_recordings'), '41_K2p3/P3_EQ.wav'))
rir_R, fs2 = sf.read(pjoin(pjoin(MPATH, 'original_recordings'), '41_K2p3/P4_EQ.wav'))

rir = np.vstack((rir_L, rir_R))

rir = rir[:, np.min((np.argmax(rir[0]), np.argmax(rir[1]))) - int(0.1*fs):]

global_max = np.max((np.abs(rir[0]), np.abs(rir[1])))

rir[0] /= np.max(np.abs(rir[0])) / global_max
rir[1] /= np.max(np.abs(rir[1])) / global_max
rir /= np.max(np.abs(rir))
sf.write(pjoin(outpath, f"{idx+3:04d}"+'_'+'41_K2p3_d1_orig.wav'), rir.T, samplerate=FS)

In [25]:

'''
# small listening test:
samples = os.listdir(outpath)

samples.sort()

x, fs = sf.read(pjoin(outpath, samples[0]))
x2, fs = sf.read(pjoin(outpath, samples[2]))

y=stereo_conv(TEST_TRACK, x.T)
y *= np.sqrt(power(TEST_TRACK) / power(y)) 
y2=stereo_conv(TEST_TRACK, x2.T)
y2 *= np.sqrt(power(TEST_TRACK) / power(y2)) 
''';

## D2, using each capsule in the Zylia recording

In [26]:
outpath = '/media/diskA/enric/parirset/d2_capsules'
if not os.path.exists(outpath):
    os.makedirs(outpath)
idx = 0
for room in trainrooms:
    for capsule in range(19):
            try:
                rir, _ = gen_out(room, 'capsules', capsule)
                sf.write(pjoin(outpath, f"{idx:04d}"+'_'+room+'_d2_cap'+ f"{capsule:02d}"+'.wav'), rir.T, samplerate=FS)
                idx += 1
            except:
                print('Error in '+str(room)+ 'in capsule '+str(capsule))

In [27]:
# small listening test:
samples = os.listdir(outpath)

samples.sort()

x, fs = sf.read(pjoin(outpath, samples[0]))
x2, fs = sf.read(pjoin(outpath, samples[15]))

y=stereo_conv(TEST_TRACK, x.T)
y *= np.sqrt(power(TEST_TRACK) / power(y)) 
y2=stereo_conv(TEST_TRACK, x2.T)
y2 *= np.sqrt(power(TEST_TRACK) / power(y2)) 


## D3, using Zylia's own Ambisonics beamformer

In [28]:
outpath = '/media/diskA/enric/parirset/d3_beamforming'
if not os.path.exists(outpath):
    os.makedirs(outpath)

In [29]:
idx = 0
for room in trainrooms:
    for capsule in range(19):
        #try:
        rir, _ = gen_out(room, 'beams', capsule)
        sf.write(pjoin(outpath, f"{idx:04d}"+'_'+room+'_d3_bf'+ f"{capsule:02d}"+'.wav'), rir.T, samplerate=FS)
        idx += 1
        #except:
        #    print('Error in '+str(room)+ 'in capsule '+str(capsule))

## D4, PERMUTE the direct sound and the reverb tail from the original recordings

In [30]:
outpath = '/media/diskA/enric/parirset/d4_permute'
if not os.path.exists(outpath):
    os.makedirs(outpath)

In [31]:
idx = 0
for delta in deltas:
    for tail in tails:
        rir = np.hstack((delta, tail))
        sf.write(pjoin(outpath, f"{idx:04d}"+'_d4_pr.wav'), rir.T, samplerate=FS)
        idx +=1

In [ ]:
'''
# small listening test:
samples = os.listdir(outpath)

samples.sort()

x, fs = sf.read(pjoin(outpath, samples[0]))
x2, fs = sf.read(pjoin(outpath, samples[15]))

y=stereo_conv(TEST_TRACK, x.T)
y *= np.sqrt(power(TEST_TRACK) / power(y)) 
y2=stereo_conv(TEST_TRACK, x2.T)
y2 *= np.sqrt(power(TEST_TRACK) / power(y2)) 
''';